In [ ]:
# ==========================================
# BLOCK 1: SMART ENVIRONMENT SETUP
# ==========================================
# @title ⚙️ 1. Setup Environment & Persistence
# @markdown ---
# @markdown ### 📂 Workspace Configuration
# @markdown Define your Google Drive workspace and optional shared drive paths below.
# @markdown ---
DRIVE_WORKSPACE_NAME = "AutoScribe_Workspace" # @param {type:"string"}
SHARED_DRIVE_NAME = "" # @param {type:"string"}

import os, sys, shutil
from datetime import datetime
from google.colab import drive
from IPython.display import clear_output, display
from IPython.utils import capture
import ipywidgets as widgets

# --- WORKSPACE PATHS ---
LOCAL_WORKSPACE = "/content/AutoScribe_Local"
TEMP_DIR = f"{LOCAL_WORKSPACE}/temp_media"
LOCAL_OUTPUT = f"{LOCAL_WORKSPACE}/outputs"
LOCAL_LOG = f"{LOCAL_WORKSPACE}/autoscribe_debug.log"
os.makedirs(TEMP_DIR, exist_ok=True)
os.makedirs(LOCAL_OUTPUT, exist_ok=True)

def log_event(level, message, print_to_console=True):
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
    log_entry = f"[{timestamp}] {level}: {message}"
    with open(LOCAL_LOG, "a", encoding="utf-8") as f:
        f.write(log_entry + "\n")
    if print_to_console:
        color = {"INFO": "32", "WARNING": "33", "ERROR": "31"}.get(level, "37")
        print(f"\033[{color}m{log_entry}\033[0m", flush=True)

def inf(msg, style, wdth):
    display(widgets.Button(description=msg, disabled=True, button_style=style, layout=widgets.Layout(min_width=wdth)))

if not os.path.exists('/content/drive/MyDrive'):
    log_event("INFO", "Mounting Google Drive...")
    drive.mount('/content/drive', force_remount=True)
else:
    log_event("INFO", "✅ Google Drive already mounted.")

root_path = f"/content/drive/Shareddrives/{SHARED_DRIVE_NAME}" if SHARED_DRIVE_NAME and os.path.exists(f"/content/drive/Shareddrives/{SHARED_DRIVE_NAME}") else "/content/drive/MyDrive"
workspace_name = DRIVE_WORKSPACE_NAME.strip() or "AutoScribe_Workspace"
DRIVE_BASE = f"{root_path}/{workspace_name}"
DRIVE_OUTPUT_DIR = os.path.join(DRIVE_BASE, "outputs")
os.makedirs(DRIVE_OUTPUT_DIR, exist_ok=True)

try:
    import faster_whisper
    log_event("INFO", "✅ Dependencies already present.")
except ImportError:
    log_event("INFO", "Installing dependencies to local NVMe...")
    with capture.capture_output() as cap:
        !pip install -q yt-dlp faster-whisper ffmpeg-python transformers accelerate torch torchvision
        !pip install -q --upgrade --no-deps yt-dlp

os.environ["HF_HOME"] = "/content/local_hf_cache"
BLOCK_1_COMPLETED = True
clear_output()
inf('\u2714 Environment Synchronized (Resilient Setup)', 'success', '400px')

In [ ]:
# ==========================================
# BLOCK 2: INGESTION & ROUTING
# ==========================================
# @title 📥 2. Source Configuration & Routing
# @markdown ---
# @markdown ### ⚙️ 1. Select Source
SOURCE_TYPE = "URL" # @param ["URL", "Google_Drive"]

# @markdown ---
# @markdown ### 🔗 Option A: URL
YOUTUBE_URL = "https://youtube.com/playlist?list=..." # @param {type:"string"}

# @markdown ### 📂 Option B: Google Drive
DRIVE_FILE_PATH = "/content/drive/MyDrive/your_video.mp4" # @param {type:"string"}

# @markdown ---
# @markdown ### 🧠 2. AI Processing Settings
WHISPER_MODE = "Auto" # @param ["On", "Off", "Auto"]
VISION_FALLBACK = True # @param {type:"boolean"}
# @markdown ---

import yt_dlp, subprocess, shutil, os
routing_queue = []

if 'BLOCK_1_COMPLETED' not in locals():
    raise RuntimeError("🚨 ERROR: Run Block 1 first.")

def analyze_and_route(file_path, title, video_id):
    log_event("INFO", f"Analyzing: {title}...")
    cmd = ["ffprobe", "-v", "error", "-select_streams", "a", "-show_entries", "stream=codec_type", "-of", "default=nw=1:nk=1", file_path]
    res = subprocess.run(cmd, stdout=subprocess.PIPE, text=True)
    has_audio = "audio" in res.stdout.lower()
    use_vision = (WHISPER_MODE == "Off") or (WHISPER_MODE == "Auto" and not has_audio)
    routing_queue.append({'id': video_id, 'title': title, 'path': file_path, 'use_vision': use_vision})

if SOURCE_TYPE == "Google_Drive" and os.path.exists(DRIVE_FILE_PATH):
    fname = os.path.basename(DRIVE_FILE_PATH)
    dest = os.path.join(TEMP_DIR, fname)
    if not os.path.exists(dest): shutil.copy2(DRIVE_FILE_PATH, dest)
    analyze_and_route(dest, fname, fname.split('.')[0])
elif SOURCE_TYPE == "URL":
    opts = {'extract_flat': 'in_playlist', 'quiet': True, 'extractor_args': {'youtube': ['player_client=android']}}
    with yt_dlp.YoutubeDL(opts) as ydl:
        try:
            info = ydl.extract_info(YOUTUBE_URL, download=False)
            entries = info['entries'] if 'entries' in info else [info]
            for e in entries:
                v_id, title = e['id'], e.get('title', e['id'])
                dl_opts = {'format': 'bestaudio/best', 'outtmpl': f'{TEMP_DIR}/{v_id}.%(ext)s', 'extractor_args': {'youtube': ['player_client=android']}, 'quiet': True}
                with yt_dlp.YoutubeDL(dl_opts) as ydl_dl:
                    loc = ydl_dl.prepare_filename(e)
                    if not os.path.exists(loc): ydl_dl.extract_info(e['url'], download=True)
                    analyze_and_route(loc, title, v_id)
        except Exception as ex: log_event("ERROR", f"URL Ingestion failed: {ex}")

BLOCK_2_COMPLETED = True
log_event("INFO", f"✅ Queue Ready: {len(routing_queue)} items.")

In [ ]:
# ==========================================
# BLOCK 3: ADAPTIVE AI PROCESSING
# ==========================================
# @title 🧠 3. Execute AI Processing
# @markdown ---
# @markdown ### 🤖 Adaptive Processing Engine
# @markdown This block performs the following automated steps:
# @markdown 1. **Duration Analysis:** Checks the length of each media file.
# @markdown 2. **Model Scaling:** Automatically selects `large-v3` or `medium` (>4h).
# @markdown 3. **VAD Filtering:** Cleans audio and reduces RAM overhead.
# @markdown 4. **Resource Management:** Flushes GPU VRAM and System RAM.
# @markdown 5. **Cloud Sync:** Securely exports results to Google Drive.
# @markdown ---

import gc, torch, subprocess, os, re
from datetime import datetime
from PIL import Image

if 'BLOCK_2_COMPLETED' not in locals():
    raise RuntimeError("🚨 ERROR: Run Block 1 and 2 first.")

from faster_whisper import WhisperModel
run_ts = datetime.now().strftime("%Y-%m-%d_%H-%M")
LOCAL_EXP = os.path.join(LOCAL_OUTPUT, f"Run_{run_ts}")
os.makedirs(LOCAL_EXP, exist_ok=True)

def get_duration(fp):
    cmd = ["ffprobe", "-v", "error", "-show_entries", "format=duration", "-of", "default=noprint_wrappers=1:nokey=1", fp]
    res = subprocess.run(cmd, stdout=subprocess.PIPE, text=True)
    return float(res.stdout)/60 if res.stdout else 0

for task in [t for t in routing_queue if not t['use_vision']]:
    dur = get_duration(task['path'])
    m_size = "medium" if dur > 240 else "large-v3"
    log_event("INFO", f"🎙️ Processing {task['title']} ({dur:.1f}m) with {m_size}...")
    
    try:
        model = WhisperModel(m_size, device="cuda", compute_type="float16", download_root=os.environ["HF_HOME"])
        segs, _ = model.transcribe(task['path'], language="sv", vad_filter=True)
        safe_name = re.sub(r'[^\w\s-]', '', task['title']).strip().replace(' ', '_')[:50]
        with open(os.path.join(LOCAL_EXP, f"{safe_name}.md"), "w", encoding="utf-8") as f:
            f.write(f"# {task['title']}\nModel: {m_size}\n\n")
            for s in segs: f.write(f"[{int(s.start)//60:02d}:{int(s.start)%60:02d}] {s.text.strip()}\n")
    except Exception as e: log_event("ERROR", f"Whisper Error: {e}")
    finally:
        if 'model' in locals(): del model
        gc.collect(); torch.cuda.empty_cache()

shutil.copytree(LOCAL_EXP, os.path.join(DRIVE_OUTPUT_DIR, f"Run_{run_ts}"))
BLOCK_3_COMPLETED = True
log_event("INFO", "✅ Block 3 Complete.")

In [ ]:
# ==========================================
# BLOCK 4: FINALIZE & DISCONNECT
# ==========================================
# @title 📝 4. Finalize & Disconnect
# @markdown ---
# @markdown ### 🧹 Housekeeping & Cleanup
# @markdown **Warning:** This will wipe the local NVMe cache, export final logs, and disconnect the session.
# @markdown ---
CONFIRM_SHUTDOWN = True # @param {type:"boolean"}

from google.colab import runtime
import shutil, os

if 'BLOCK_3_COMPLETED' in locals() and CONFIRM_SHUTDOWN:
    if os.path.exists(LOCAL_WORKSPACE): shutil.rmtree(LOCAL_WORKSPACE)
    print("🎉 Workspace cleared. Disconnecting session...")
    runtime.unassign()
else:
    print("⚠️ Shutdown aborted.")